In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()

model = ChatGroq(model="openai/gpt-oss-120b")

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content

    return {'joke': response}

In [6]:

def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = model.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the dough was rising, the crust was getting a promotion, and it finally wanted to *deliver* on its “cheese” potential! 🍕😄',
 'explanation': '**Explanation of the Joke**\n\n> **Why did the pizza apply for a job?**  \n> **Because it heard the dough was rising, the crust was getting a promotion, and it finally wanted to *deliver* on its “cheese” potential! 🍕😄**\n\n---\n\n### 1. The Set‑up  \nThe opening line (“Why did the pizza apply for a job?”) is a classic “Why‑did‑the‑X…?” set‑up that prepares the listener for a punch‑line that explains an absurd or unexpected reason. In this case the “X” is a pizza, an inanimate food item, which already signals that the answer will be playful and not literal.\n\n### 2. The Punch‑line – A String of Food‑Related Puns  \n\n| Phrase in the punch‑line | Literal meaning (food context) | Figurative meaning (work‑place jargon) | Why it’s funny |\n|--------------------------

In [9]:
workflow.get_state(config1) #sanpshot of the final state

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the dough was rising, the crust was getting a promotion, and it finally wanted to *deliver* on its “cheese” potential! 🍕😄', 'explanation': '**Explanation of the Joke**\n\n> **Why did the pizza apply for a job?**  \n> **Because it heard the dough was rising, the crust was getting a promotion, and it finally wanted to *deliver* on its “cheese” potential! 🍕😄**\n\n---\n\n### 1. The Set‑up  \nThe opening line (“Why did the pizza apply for a job?”) is a classic “Why‑did‑the‑X…?” set‑up that prepares the listener for a punch‑line that explains an absurd or unexpected reason. In this case the “X” is a pizza, an inanimate food item, which already signals that the answer will be playful and not literal.\n\n### 2. The Punch‑line – A String of Food‑Related Puns  \n\n| Phrase in the punch‑line | Literal meaning (food context) | Figurative meaning (work‑place jargon) | Why it’s funny |\n|-------

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the dough was rising, the crust was getting a promotion, and it finally wanted to *deliver* on its “cheese” potential! 🍕😄', 'explanation': '**Explanation of the Joke**\n\n> **Why did the pizza apply for a job?**  \n> **Because it heard the dough was rising, the crust was getting a promotion, and it finally wanted to *deliver* on its “cheese” potential! 🍕😄**\n\n---\n\n### 1. The Set‑up  \nThe opening line (“Why did the pizza apply for a job?”) is a classic “Why‑did‑the‑X…?” set‑up that prepares the listener for a punch‑line that explains an absurd or unexpected reason. In this case the “X” is a pizza, an inanimate food item, which already signals that the answer will be playful and not literal.\n\n### 2. The Punch‑line – A String of Food‑Related Puns  \n\n| Phrase in the punch‑line | Literal meaning (food context) | Figurative meaning (work‑place jargon) | Why it’s funny |\n|------

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti get a promotion?\n\nBecause it always *pasta*ble to rise to the occasion and never gets tangled up in office drama! 🍝😄',
 'explanation': '**Explanation of the joke**\n\n> **Why did the spaghetti get a promotion?**  \n> **Because it always *pasta*ble to rise to the occasion and never gets tangled up in office drama! 🍝😄**\n\n---\n\n### 1. The core of the humor: wordplay (puns)\n\n| Phrase in the punchline | What it sounds like / what it actually means |\n|--------------------------|----------------------------------------------|\n| **“pasta‑ble”** | A play on the word **“passable”** (meaning “acceptable” or “good enough”). By swapping “pass” for the food‑related “pasta,” the joke turns a normal compliment into a food‑themed one. |\n| **“rise to the occasion”** | A common idiom meaning “to meet a challenge successfully.” In cooking, pasta (especially fresh dough) *rises* when it’s boiled or when the dough is left to ferment, so the phrase

In [12]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti get a promotion?\n\nBecause it always *pasta*ble to rise to the occasion and never gets tangled up in office drama! 🍝😄', 'explanation': '**Explanation of the joke**\n\n> **Why did the spaghetti get a promotion?**  \n> **Because it always *pasta*ble to rise to the occasion and never gets tangled up in office drama! 🍝😄**\n\n---\n\n### 1. The core of the humor: wordplay (puns)\n\n| Phrase in the punchline | What it sounds like / what it actually means |\n|--------------------------|----------------------------------------------|\n| **“pasta‑ble”** | A play on the word **“passable”** (meaning “acceptable” or “good enough”). By swapping “pass” for the food‑related “pasta,” the joke turns a normal compliment into a food‑themed one. |\n| **“rise to the occasion”** | A common idiom meaning “to meet a challenge successfully.” In cooking, pasta (especially fresh dough) *rises* when it’s boiled or when the dough is left to fer

In [13]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti get a promotion?\n\nBecause it always *pasta*ble to rise to the occasion and never gets tangled up in office drama! 🍝😄', 'explanation': '**Explanation of the joke**\n\n> **Why did the spaghetti get a promotion?**  \n> **Because it always *pasta*ble to rise to the occasion and never gets tangled up in office drama! 🍝😄**\n\n---\n\n### 1. The core of the humor: wordplay (puns)\n\n| Phrase in the punchline | What it sounds like / what it actually means |\n|--------------------------|----------------------------------------------|\n| **“pasta‑ble”** | A play on the word **“passable”** (meaning “acceptable” or “good enough”). By swapping “pass” for the food‑related “pasta,” the joke turns a normal compliment into a food‑themed one. |\n| **“rise to the occasion”** | A common idiom meaning “to meet a challenge successfully.” In cooking, pasta (especially fresh dough) *rises* when it’s boiled or when the dough is left to fe

## Time Travel

In [17]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1cf0d7-885f-678a-8000-f21e2177f994"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1cf0d7-885f-678a-8000-f21e2177f994'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-10-23T18:13:38.876597+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1cf0d7-885d-639a-bfff-0a0da64f06ee'}}, tasks=(PregelTask(id='242e49fa-4804-a9bc-b933-3535065f871f', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the dough was rising, the crust was getting a promotion, and it finally wanted to *deliver* on its “cheese” potential! 🍕😄'}),), interrupts=())

In [19]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1cf0d7-885f-678a-8000-f21e2177f994"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the company was looking for someone with *great* "crust"‑ometrics and a lot of "topping" potential! 🍕😄',
 'explanation': '**Joke:**  \n*Why did the pizza apply for a job?*  \n*Because it heard the company was looking for someone with **great “crust‑ometrics”** and a lot of **“topping” potential!** 🍕😄*\n\n---\n\n## How the joke works  \n\n| Element of the joke | What it sounds like / what it really means | Why it’s funny |\n|---------------------|---------------------------------------------|----------------|\n| **“crust‑ometrics”** | A mash‑up of **crust** (the outer part of a pizza) + **metrics** (measurements, data, statistics). It sounds like “*statistics*” or “*metrics*”. | The word “crust‑ometrics” pretends that a pizza can be evaluated with the same kind of quantitative analysis we use for people or businesses. The absurdity of a food item having a professional‑sounding specialty makes us laugh.

In [20]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the company was looking for someone with *great* "crust"‑ometrics and a lot of "topping" potential! 🍕😄', 'explanation': '**Joke:**  \n*Why did the pizza apply for a job?*  \n*Because it heard the company was looking for someone with **great “crust‑ometrics”** and a lot of **“topping” potential!** 🍕😄*\n\n---\n\n## How the joke works  \n\n| Element of the joke | What it sounds like / what it really means | Why it’s funny |\n|---------------------|---------------------------------------------|----------------|\n| **“crust‑ometrics”** | A mash‑up of **crust** (the outer part of a pizza) + **metrics** (measurements, data, statistics). It sounds like “*statistics*” or “*metrics*”. | The word “crust‑ometrics” pretends that a pizza can be evaluated with the same kind of quantitative analysis we use for people or businesses. The absurdity of a food item having a professional‑sounding speci